In [163]:
# # clube entery_age 

# age = int(input("Enter your age: "))

# def check_age(age):
#     if age < 18:
#         print("You are not allowed to enter the club.")
#     elif age >= 18 and age < 21:
#         print("You are allowed to enter the club, but you cannot drink alcohol.")
#     else:
#         print("You are allowed to enter the club and drink alcohol.")

# check_age(age)

In [164]:
def even_or_odd(numbers):
    numbers = [4,2,13,5,1,7,9]
    even_count = 0
    odd_count = 0
    for i in numbers:
        if i % 2 == 0:
            print(f"{i} is even")
            even_count += 1
        else:
            print(f"{i} is odd")
            odd_count += 1
    print(f"Total even numbers: {even_count}")
    print(f"Total odd numbers: {odd_count}")

even_or_odd(numbers)

4 is even
2 is even
13 is odd
5 is odd
1 is odd
7 is odd
9 is odd
Total even numbers: 2
Total odd numbers: 5


In [165]:
numbers = [4,2,13,5,1,7,9]
new_list = []

def sorted_numbers():
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] > numbers[j]:
                numbers[i], numbers[j] = numbers[j], numbers[i]
    return numbers

soreted_numbers()

[1, 2, 4, 5, 7, 9, 13]

In [166]:
# # can person corssthe road? if the person is standing on the other side of the road and the traffic light is red, then the person can cross the road. 
# # If the traffic light is green, then the person should wait until it turns red before crossing.

# traffic_light = input("Enter the color of the traffic light (red/green): ").lower()
# if traffic_light == "red":
#     print("You can cross the road.")
# elif traffic_light == "green":
#     print("You should wait until the light turns red before crossing.")
# elif traffic_light == "yellow":
#     print("You should wait until the light turns red before crossing.")
# elif traffic_light == "blue":
#     print("Blue is not a valid traffic light color. Please enter red, green, or yellow.")
# else:
#     print("Invalid input. Please enter red, green, or yellow.")

In [167]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [168]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   
print(f"Using device: {device}")

Using device: cuda


In [169]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN,self).__init__()

        self.f1 = nn.Linear(10, 5)  
        self.f2 = nn.Linear(5, 1)

        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()

        self.f3 = nn.Linear(20, 10)  
        self.f4 = nn.Linear(10, 5)
        
        self.dropout = nn.Dropout(0.5)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.f1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.f2(x)
        return x
    


model = SimpleNN().to(device)

model



SimpleNN(
  (f1): Linear(in_features=10, out_features=5, bias=True)
  (f2): Linear(in_features=5, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (relu): ReLU()
  (f3): Linear(in_features=20, out_features=10, bias=True)
  (f4): Linear(in_features=10, out_features=5, bias=True)
)

In [170]:
# now we are going to buid the attention all yoou nedd model

class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert (self.head_dim * heads == embed_size), "Embedding size needs to be divisible by heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        # Split the embedding into `self.heads` pieces
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.embed_size ** (1 / 2)), dim=3)

        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        out = self.fc_out(out)
        return out

In [171]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads, dropout, forward_expansion):
        super().__init__()

        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask):
        attention = self.attention(value, key, query, mask)
        x = self.dropout(self.norm1(attention + query))

        forward = self.feed_forward(x)
        return self.dropout(self.norm2(forward + x))

In [172]:
class Encoder(nn.Module):
    def __init__(
        self, src_vocab_size, embed_size, num_layers, heads, device,
        forward_expansion, dropout, max_length
    ):
        super().__init__()

        self.device = device
        self.word_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            TransformerBlock(
                embed_size, heads, dropout, forward_expansion
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, mask):
        n, seq_length = x.shape
        positions = torch.arange(seq_length, device=self.device)
        positions = positions.unsqueeze(0).expand(n, seq_length)

        out = self.dropout(
            self.word_embedding(x) + self.position_embedding(positions)
        )

        for layer in self.layers:
            out = layer(out, out, out, mask)

        return out

In [173]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_size, heads, forward_expansion, dropout, device):
        super().__init__()

        self.attention = SelfAttention(embed_size, heads)
        self.norm = nn.LayerNorm(embed_size)
        self.transformer_block = TransformerBlock(
            embed_size, heads, dropout, forward_expansion
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, trg_mask):
        attention = self.attention(x, x, x, trg_mask)
        query = self.dropout(self.norm(attention + x))

        return self.transformer_block(value, key, query, src_mask)




In [174]:
class Decoder(nn.Module):
    def __init__(
        self, trg_vocab_size, embed_size, num_layers, heads,
        forward_expansion, dropout, device, max_length
    ):
        super().__init__()

        self.device = device
        self.word_embedding = nn.Embedding(trg_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderBlock(
                embed_size, heads, forward_expansion, dropout, device
            )
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(embed_size, trg_vocab_size)

    def forward(self, x, enc_out, src_mask, trg_mask):
        n, seq_length = x.shape
        positions = torch.arange(seq_length, device=self.device)
        positions = positions.unsqueeze(0).expand(n, seq_length)

        x = self.dropout(
            self.word_embedding(x) + self.position_embedding(positions)
        )

        for layer in self.layers:
            x = layer(x, enc_out, enc_out, src_mask, trg_mask)

        return self.fc_out(x)


In [175]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, src_pad_idx, trg_pad_idx, embed_size =256, num_layers = 6, forward_expansion = 4, heads = 8, dropout = 0.1, max_length = 100):
        super(Transformer,self).__init__()

        self.encoder = Encoder(
            src_vocab_size,
            embed_size,
            num_layers,
            heads,
            device,
            forward_expansion,
            dropout,
            max_length
        )

        self.decoder = Decoder(
            trg_vocab_size,
            embed_size,
            num_layers,
            heads,
            forward_expansion,
            dropout,
            device,
            max_length
        )

        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

    def make_src_mask(self, src):
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

        return src_mask.to(self.device)

    def make_trg_mask(self, trg):
        N, trg_len = trg.shape
        trg_mask = torch.tril(torch.ones((trg_len, trg_len))).expand(N, 1, trg_len, trg_len)

        return trg_mask.to(self.device)

    def forward(self, src, trg):

        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)

        enc_src = self.encoder(src, src_mask)
        out = self.decoder(trg, enc_src, src_mask, trg_mask)

        return out

In [176]:
if __name__ == "__main__":

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    x = torch.tensor([[1, 5, 6, 4, 3, 9, 5, 2, 0], [1, 8, 7, 3, 4, 5, 6, 7, 2]]).to(device)

    trg = torch.tensor([[1, 7, 4, 3, 5, 9, 2, 0], [1, 5, 6, 2, 4, 5, 7, 2]]).to(device)

    scr_pad_idx = 0
    trg_pad_idx = 0
    scr_vocab_size = 10
    trg_vocab_size = 10

    model = Transformer(src_vocab_size=scr_vocab_size, trg_vocab_size=trg_vocab_size, src_pad_idx=scr_pad_idx, trg_pad_idx=trg_pad_idx).to(device)

    out = model(x, trg[:, :-1])

    print(out.shape)

torch.Size([2, 7, 10])


In [177]:
model = Transformer(
    src_vocab_size=scr_vocab_size,
    trg_vocab_size=trg_vocab_size,
    src_pad_idx=scr_pad_idx,
    trg_pad_idx=trg_pad_idx,
).to(device)

model.eval()

with torch.no_grad():
    out = model(x, trg[:, :-1])

print(out.shape)

torch.Size([2, 7, 10])
